# Task 1: Data Exploration and Cleaning

**Project:** Drug Review Classification — Condition Prediction from Patient Reviews  
**Dataset:** UCI Drug Review Dataset (Drugs.com)  
**Business Objective:** Classify patient conditions (Depression, High Blood Pressure, Type 2 Diabetes) from drug review text using machine learning.

---

## Overview

This notebook covers:
1. Data gathering and initial inspection
2. Missing value analysis
3. Data cleaning steps
4. Exploratory Data Analysis (EDA)
5. Class imbalance analysis
6. Outlier detection
7. Key findings and summary

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import html
import re
from pathlib import Path

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PALETTE = ['#14b8a6', '#eaa0a2', '#67cbdc', '#3a4664', '#cbcbcb']

print('Libraries loaded successfully.')

In [ ]:
# Load raw training and test datasets
df_raw_train = pd.read_csv('../data/raw/drugsComTrain_raw.csv')
df_raw_test  = pd.read_csv('../data/raw/drugsComTest_raw.csv')

print('=== RAW TRAINING DATASET ===')
print(f'Shape      : {df_raw_train.shape}')
print(f'Columns    : {list(df_raw_train.columns)}')
print(f'\nFirst 3 rows:')
df_raw_train.head(3)

## 2. Dataset Description

The UCI Drug Review dataset (Drugs.com) contains patient reviews of specific medications along with:

| Column | Description |
|--------|-------------|
| `uniqueID` | Unique review identifier |
| `drugName` | Name of the drug being reviewed |
| `condition` | Medical condition the drug was taken for |
| `review` | Patient review text |
| `rating` | Patient satisfaction rating (1–10) |
| `date` | Date the review was posted |
| `usefulCount` | Number of users who found the review helpful |

The dataset covers **161,297 reviews** across many conditions in the full version. For this project, we filter to **three target conditions** aligned with our business objective.

## 3. Missing Value Analysis

In [ ]:
print('=== MISSING VALUES — TRAINING SET ===')
missing = df_raw_train.isnull().sum()
missing_pct = (missing / len(df_raw_train) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
print(missing_df)

if missing.sum() == 0:
    print('\n✓ No missing values found in the raw training dataset.')
else:
    print(f'\n⚠ Total missing values: {missing.sum()}')

In [ ]:
# Data types and basic info
print('=== DATA TYPES ===')
print(df_raw_train.dtypes)
print(f'\nTotal records (train): {len(df_raw_train):,}')
print(f'Total records (test) : {len(df_raw_test):,}')
print(f'Unique conditions (train): {df_raw_train["condition"].nunique():,}')
print(f'Unique drugs (train)     : {df_raw_train["drugName"].nunique():,}')

## 4. Data Cleaning

Three cleaning steps were applied:

1. **HTML entity decoding** — reviews contain encoded characters such as `&#039;` (apostrophe) and `&amp;` (ampersand) from web scraping. These are decoded to their plain-text equivalents.
2. **Condition standardisation** — condition names contain spelling variations (e.g. `type 2 diabetes` vs `Diabetes, Type 2`). These are normalised to consistent labels.
3. **Date parsing** — dates stored as strings (`28-Feb-12`) are converted to `datetime` objects for temporal analysis.

In [ ]:
# Load cleaned dataset (output of scripts/clean_train_data.py)
df_train = pd.read_csv('../data/processed/cleaned_train_data.csv', parse_dates=['review_date'])
df_test  = pd.read_csv('../data/processed/cleaned_test_data.csv',  parse_dates=['review_date'])

print('=== CLEANED TRAINING DATASET ===')
print(f'Shape   : {df_train.shape}')
print(f'Columns : {list(df_train.columns)}')
print(f'\nDate range : {df_train["review_date"].min().date()} to {df_train["review_date"].max().date()}')
df_train.head(3)

In [ ]:
# Demonstrate HTML cleaning effect
raw_sample = df_raw_train['review'].iloc[0]
clean_sample = df_train['review'].iloc[0]

print('BEFORE cleaning (first 200 chars):')
print(repr(raw_sample[:200]))
print('\nAFTER cleaning (first 200 chars):')
print(repr(clean_sample[:200]))

## 5. Target Condition Distribution — Class Imbalance Analysis

Understanding the class distribution is critical before model training. An imbalanced dataset can cause a model to be biased towards the majority class, producing misleadingly high accuracy while performing poorly on minority classes.

In [ ]:
condition_counts = df_train['condition'].value_counts()
condition_pct    = (condition_counts / len(df_train) * 100).round(1)

print('=== CLASS DISTRIBUTION ===')
for cond in condition_counts.index:
    bar = '█' * int(condition_pct[cond] / 2)
    print(f'  {cond:<25} {condition_counts[cond]:>5,}  ({condition_pct[cond]:>5.1f}%)  {bar}')

imbalance_ratio = condition_counts.max() / condition_counts.min()
print(f'\nImbalance ratio (max/min): {imbalance_ratio:.1f}:1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
bars = axes[0].bar(condition_counts.index, condition_counts.values,
                   color=PALETTE[:3], edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_title('Class Distribution — Target Conditions', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Number of Reviews', fontsize=12)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', labelsize=11)
for bar, count, pct in zip(bars, condition_counts.values, condition_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{count:,}\n({pct}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, condition_counts.max() * 1.18)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    condition_counts.values,
    labels=condition_counts.index,
    colors=PALETTE[:3],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title('Proportion of Each Condition', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: outputs/figures/class_distribution.png')

### Class Imbalance — Findings and Mitigation

The dataset exhibits **significant class imbalance**:

- **Depression** dominates at **64.9%** (9,154 reviews)
- **Diabetes, Type 2** accounts for **18.1%** (2,554 reviews)  
- **High Blood Pressure** accounts for **17.0%** (2,403 reviews)

The imbalance ratio is approximately **3.8:1** (Depression vs High Blood Pressure).

**Mitigation strategy:** All classifiers in the ensemble use `class_weight='balanced'`, which automatically adjusts the contribution of each class during training, inversely proportional to its frequency. This prevents the model from simply predicting Depression for every input.

## 6. Rating Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Overall rating distribution
rating_counts = df_train['rating'].value_counts().sort_index()
colors = ['#ef4444' if r <= 3 else '#eab308' if r <= 6 else '#22c55e'
          for r in rating_counts.index]
bars = axes[0].bar(rating_counts.index, rating_counts.values,
                   color=colors, edgecolor='white', linewidth=1)
axes[0].set_title('Overall Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating (1–10)', fontsize=12)
axes[0].set_ylabel('Number of Reviews', fontsize=12)
axes[0].set_xticks(range(1, 11))
legend_patches = [
    mpatches.Patch(color='#ef4444', label='Negative (1–3)'),
    mpatches.Patch(color='#eab308', label='Neutral (4–6)'),
    mpatches.Patch(color='#22c55e', label='Positive (7–10)'),
]
axes[0].legend(handles=legend_patches, fontsize=10)

# Average rating by condition
avg_ratings = df_train.groupby('condition')['rating'].mean().sort_values(ascending=False)
bars2 = axes[1].barh(avg_ratings.index, avg_ratings.values,
                     color=PALETTE[:3], edgecolor='white', height=0.5)
axes[1].set_title('Average Rating by Condition', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Rating', fontsize=12)
axes[1].set_xlim(0, 10.5)
for bar, val in zip(bars2, avg_ratings.values):
    axes[1].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRating statistics:')
print(df_train['rating'].describe().round(2))
pos_pct = (df_train['rating'] >= 7).mean() * 100
neg_pct = (df_train['rating'] <= 3).mean() * 100
print(f'\nPositive reviews (rating ≥ 7): {pos_pct:.1f}%')
print(f'Negative reviews (rating ≤ 3): {neg_pct:.1f}%')

### Rating Distribution — Findings

The rating distribution is **heavily right-skewed**, with ratings of 9 and 10 being the most common. This suggests that patients who post reviews tend to do so when they have had strongly positive or strongly negative experiences — neutral experiences are underrepresented.

Key observations:
- Over **57%** of reviews have a rating of 7 or higher (positive)
- Depression reviews have the lowest average rating, reflecting the complexity of treatment
- Rating is not used as a direct feature for classification — it is metadata that informs drug recommendations

## 7. Review Length Analysis

In [ ]:
df_train['review_length'] = df_train['review'].str.split().str.len()
df_test['review_length']  = df_test['review'].str.split().str.len()

print('=== REVIEW LENGTH STATISTICS ===')
print(df_train['review_length'].describe().round(1))

print('\nBy condition:')
print(df_train.groupby('condition')['review_length'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Overall review length histogram
axes[0].hist(df_train['review_length'].clip(upper=400), bins=50,
             color='#14b8a6', edgecolor='white', alpha=0.85)
axes[0].axvline(df_train['review_length'].median(), color='#ef4444',
                linestyle='--', linewidth=2, label=f'Median: {df_train["review_length"].median():.0f} words')
axes[0].axvline(df_train['review_length'].mean(), color='#3b82f6',
                linestyle='--', linewidth=2, label=f'Mean: {df_train["review_length"].mean():.0f} words')
axes[0].set_title('Review Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Word Count (clipped at 400)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=11)

# Review length by condition — box plot
conditions = df_train['condition'].unique()
data_by_cond = [df_train[df_train['condition'] == c]['review_length'].clip(upper=400).values
                for c in conditions]
bp = axes[1].boxplot(data_by_cond, labels=conditions, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], PALETTE[:3]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[1].set_title('Review Length by Condition', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Word Count (clipped at 400)', fontsize=12)
axes[1].tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/review_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Review Length — Findings

- Average review length is approximately **83 words**
- Depression reviews tend to be longer, reflecting the emotional and detailed nature of mental health experiences
- The distribution is right-skewed — some reviews exceed 1,000 words while many are under 30 words
- Reviews under 23 words (10th percentile) are flagged as potentially insufficient for reliable classification and handled by the OOD detection system

## 8. Useful Count — Outlier Detection (IQR Method)

In [ ]:
Q1  = df_train['useful_count'].quantile(0.25)
Q3  = df_train['useful_count'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_train[(df_train['useful_count'] < lower_bound) |
                    (df_train['useful_count'] > upper_bound)]

print('=== USEFUL COUNT — IQR OUTLIER DETECTION ===')
print(f'Q1           : {Q1:.2f}')
print(f'Q3           : {Q3:.2f}')
print(f'IQR          : {IQR:.2f}')
print(f'Lower bound  : {lower_bound:.2f}')
print(f'Upper bound  : {upper_bound:.2f}')
print(f'Outliers     : {len(outliers):,} ({len(outliers)/len(df_train)*100:.1f}%)')
print(f'Max value    : {df_train["useful_count"].max():,}')
print(f'Mean         : {df_train["useful_count"].mean():.1f}')
print(f'Median       : {df_train["useful_count"].median():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution (clipped)
axes[0].hist(df_train['useful_count'].clip(upper=300), bins=60,
             color='#67cbdc', edgecolor='white', alpha=0.85)
axes[0].axvline(upper_bound, color='#ef4444', linestyle='--', linewidth=2,
                label=f'IQR upper bound: {upper_bound:.0f}')
axes[0].set_title('Useful Count Distribution (clipped at 300)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Useful Count', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend(fontsize=10)

# Log-transformed
axes[1].hist(np.log1p(df_train['useful_count']), bins=50,
             color='#eaa0a2', edgecolor='white', alpha=0.85)
axes[1].set_title('Useful Count — Log-Transformed (log1p)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + Useful Count)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/figures/useful_count_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

### Outlier Handling Decision

The IQR method identifies reviews with very high useful counts as statistical outliers (upper bound ≈ 100). However, **these outliers are not removed** for the following reasons:

1. A high useful count indicates the review was particularly informative — removing it would discard valuable signal
2. The useful count is not used directly as a classification feature — it is used only in the drug recommendation scoring
3. A **log transformation** (`log1p`) is applied during feature engineering, which compresses the outlier values and reduces their disproportionate influence

## 9. Temporal Analysis

In [ ]:
df_train['review_year'] = df_train['review_date'].dt.year

reviews_per_year = df_train.groupby('review_year').size()
reviews_by_cond_year = df_train.groupby(['review_year', 'condition']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total reviews per year
axes[0].bar(reviews_per_year.index, reviews_per_year.values,
            color='#14b8a6', edgecolor='white', linewidth=1)
axes[0].set_title('Reviews per Year', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year', fontsize=11)
axes[0].set_ylabel('Number of Reviews', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

# Reviews per year by condition
for cond, color in zip(reviews_by_cond_year.columns, PALETTE):
    axes[1].plot(reviews_by_cond_year.index, reviews_by_cond_year[cond],
                 marker='o', linewidth=2.5, markersize=5, label=cond, color=color)
axes[1].set_title('Reviews per Year by Condition', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year', fontsize=11)
axes[1].set_ylabel('Number of Reviews', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/figures/temporal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Date range: {df_train["review_date"].min().date()} to {df_train["review_date"].max().date()}')
print(f'Years covered: {df_train["review_year"].nunique()}')

## 10. Top Drugs per Condition

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

conditions = ['Depression', 'Diabetes, Type 2', 'High Blood Pressure']

for ax, condition, color in zip(axes, conditions, PALETTE):
    top_drugs = (df_train[df_train['condition'] == condition]
                 .groupby('drug_name')['rating'].agg(['mean', 'count'])
                 .query('count >= 10')
                 .sort_values('mean', ascending=False)
                 .head(8))

    bars = ax.barh(top_drugs.index, top_drugs['mean'],
                   color=color, edgecolor='white', alpha=0.85, height=0.6)
    ax.set_xlim(0, 11)
    ax.set_title(f'Top Drugs — {condition}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Average Rating', fontsize=10)
    for bar, val in zip(bars, top_drugs['mean']):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9)
    ax.invert_yaxis()

plt.suptitle('Top-Rated Drugs per Condition (min. 10 reviews)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/top_drugs_per_condition.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Descriptive Statistics Summary

In [ ]:
print('=' * 65)
print('DESCRIPTIVE STATISTICS SUMMARY')
print('=' * 65)

print(f"""
Dataset Overview
  Training samples : {len(df_train):,}
  Test samples     : {len(df_test):,}
  Total            : {len(df_train) + len(df_test):,}
  Unique drugs     : {df_train['drug_name'].nunique():,}
  Date range       : {df_train['review_date'].min().date()} — {df_train['review_date'].max().date()}

Class Distribution
  Depression        : {condition_counts['Depression']:,}  ({condition_counts['Depression']/len(df_train)*100:.1f}%)
  Diabetes, Type 2  : {condition_counts['Diabetes, Type 2']:,}   ({condition_counts['Diabetes, Type 2']/len(df_train)*100:.1f}%)
  High Blood Pressure: {condition_counts['High Blood Pressure']:,}  ({condition_counts['High Blood Pressure']/len(df_train)*100:.1f}%)
  Imbalance ratio   : {imbalance_ratio:.1f}:1

Rating Statistics
  Mean   : {df_train['rating'].mean():.2f} / 10
  Median : {df_train['rating'].median():.1f} / 10
  Positive (≥7) : {(df_train['rating'] >= 7).mean()*100:.1f}%
  Negative (≤3) : {(df_train['rating'] <= 3).mean()*100:.1f}%

Review Length
  Mean   : {df_train['review_length'].mean():.1f} words
  Median : {df_train['review_length'].median():.0f} words
  Min    : {df_train['review_length'].min()} words
  Max    : {df_train['review_length'].max():,} words

Useful Count
  Mean   : {df_train['useful_count'].mean():.1f}
  Median : {df_train['useful_count'].median():.0f}
  Max    : {df_train['useful_count'].max():,}
  IQR outliers : {len(outliers):,} ({len(outliers)/len(df_train)*100:.1f}%)
""")

## 12. Key Findings and Implications for Modelling

| Finding | Implication |
|---------|-------------|
| **64.9% Depression, 18.1% Diabetes, 17.0% High BP** | Class imbalance requires `class_weight='balanced'` in all classifiers |
| **Right-skewed ratings** (majority 8–10) | Rating alone is insufficient for classification; review text is the primary signal |
| **Average review length = 83 words** | Reviews are sufficiently detailed for TF-IDF and sentiment analysis |
| **Useful count highly skewed** (max = 1,291) | Log transformation (`log1p`) applied during feature engineering |
| **Reviews span 2008–2017** | No concept drift issues within this range; temporal features (year, month) included |
| **361 unique drugs** | Drug name alone is too sparse for reliable classification; review text is preferred |
| **No missing values** | No imputation required |

These findings directly informed the feature engineering strategy in Task 2 and the model selection in Tasks 3–5.